In [1]:
# From Bob: I uncommented this line or else I cannot query the data in block 3.
from project import icu_query

In [17]:
from project import bucket_ecg_report_0, simplify_race, simplify_careunit
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
original_query = icu_query()

/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [37]:
df = original_query
df.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,ecg_time,report_0
0,19942060,26995122,32178416,Neurology,Neurology,2161-01-11 16:09:00,2161-02-08 22:05:32,28.247593,2161-01-11 15:11:00,2161-02-10 13:41:00,...,40,168,186,268,484,43,-44,119,2161-01-12 08:49:00,sinus tachycardia with sinus arrhythmia
1,17945971,25486527,36652195,Med/Surg,Med/Surg,2130-03-17 04:25:01,2130-03-18 15:04:34,1.444132,2130-03-16 19:53:00,2130-03-20 16:10:00,...,40,156,194,272,658,42,-3,64,2130-03-17 05:31:00,sinus rhythm
2,13570398,29201840,30059691,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2182-10-19 17:28:12,2182-10-27 20:38:37,8.132234,2182-10-18 12:17:00,2182-10-28 14:10:00,...,40,136,202,298,636,18,0,83,2182-10-26 07:26:00,sinus rhythm
3,12506390,21301912,30062923,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2160-10-17 13:39:06,2160-10-18 14:19:00,1.027708,2160-10-16 10:30:00,2160-10-25 14:05:00,...,40,146,204,344,642,13,-37,154,2160-10-17 17:30:00,sinus rhythm
4,14178148,28718602,30126048,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2142-09-14 03:01:44,2142-09-16 19:13:11,2.674618,2142-09-13 22:53:00,2142-09-16 15:18:00,...,40,178,242,348,622,60,36,38,2142-09-14 20:13:00,sinus rhythm


In [38]:
# need to bucket report_0...
# getting frequency of report_0 codes
original_ecg_report0_frequency = original_query['report_0'].value_counts()
original_ecg_report0_frequency.to_csv('sanity_outputs/original_ecg_report0_frequency.csv')

In [39]:
# sinus rhythm, sinus tachycardia, a.fib
df['report_0'] = df['report_0'].str.lower().str.removesuffix(".").str.replace('"', '')
df = df[~df['report_0'].str.contains('warning')]
initial_drop = df['report_0'].value_counts()
initial_drop.to_csv('sanity_outputs/initial_drop.csv')
# handles: Sinus rhythm, Sinus tachycardia, Atrial fibrillation, Sinus bradycardia that had '.' suffix
# changes all to lower, drops records with 'warning' because those signals could harm our model
df = df[df['report_0'].notna()] # handles any NA values

In [40]:
# check if any NAs, there were none
print(original_query.shape)
print(df.shape)

(35474, 33)
(34830, 33)


In [41]:
# check mortality captured by these first four catgories ()
df_first_cats = df[df['report_0'].isin(['sinus rhythm', 'sinus tachycardia', 'atrial fibrillation', 'sinus bradycardia'])]
print(f"")
print(f"Mortalities captured by the first four report_0 buckets: {sum(df_first_cats['hospital_expire_flag'])} ({sum(df_first_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")


Mortalities captured by the first four report_0 buckets: 2061 (49.13%)
Total mortalities: 4195


In [24]:
df['ecg_bucket'] = df['report_0'].apply(bucket_ecg_report_0)
# check mortality captured after bucket_ecg_report_0 function
df_cats = df[df['ecg_bucket'].isin(['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional'])]
print(f"Mortalities captured by the current buckets: {sum(df_cats['hospital_expire_flag'])} ({sum(df_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")
post_bucket_function = df['ecg_bucket'].value_counts()
post_bucket_function.to_csv('sanity_outputs/post_bucket_function.csv')
# capturing 98.7% of mortalities in all categories != 'other'

Mortalities captured by the current buckets: 4141 (98.71%)
Total mortalities: 4195


In [25]:
named_buckets = ['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional']
df_other = df[~df['ecg_bucket'].isin(named_buckets)]
df_other.groupby('ecg_bucket')['hospital_expire_flag'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)

,sum,count,mean
ecg_bucket,,,
undetermined rhythm,21,143,0.146853
a-v dissociation,6,69,0.086957
wide qrs tachycardia,4,17,0.235294
"age not entered, assumed to be 50 years old for purpose of ecg interpretation",4,42,0.095238
probable junctional rhythm,4,58,0.068966
--- pediatric criteria used ---,2,31,0.064516
a-v dissociation with unclassified aberrant complexes,2,6,0.333333
probable ventricular tachycardia,2,12,0.166667
possible junctional rhythm,2,16,0.125


In [26]:
df['ecg_bucket'] = df['ecg_bucket'].where(df['ecg_bucket'].isin(named_buckets), 'other')
df['ecg_bucket'].value_counts()

ecg_bucket
normal_sinus              15670
sinus_tachy                4256
afib                       2579
sinus_brady                2508
afib_rvr                   2214
pvc                        2164
pac                        1494
paced                      1129
stemi_alert                 795
atrial_ectopic              566
other                       471
supraventricular            373
accelerated_junctional      327
idioventricular             284
Name: count, dtype: int64

In [27]:
df.to_csv('sanity_outputs/report0_cleaned.csv')
# report_0 cleaned values in new column: ecg_bucket

In [ ]:
# set gender to 0 (Male), 1 (Female)
df['gender'] = df['gender'].replace(['M', 'F'], [0,1])

/var/folders/6h/ch1ltdnj3v33nkw2k13k_h8w0000gn/T/ipykernel_8026/1892171098.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['gender'] = df['gender'].replace(['M', 'F'], [0,1])


In [29]:
# cleaning ecg measurements: explore columns
ecg_intervals = ['rr_interval','p_onset', 'p_end', 'qrs_onset', 'qrs_end', 't_end']
ecg_axes = ['p_axis', 'qrs_axis', 't_axis']

display(df[ecg_intervals].describe())
display(df[ecg_axes].describe())

,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end
count,34830.0,34830.0,34830.0,34830.0,34830.0,34830.0
mean,758.970112,6748.931467,8278.493913,211.397962,312.926357,604.997387
std,306.037663,12472.402043,13292.490344,176.20035,178.651299,188.8187
min,285.0,7.0,5.0,24.0,0.0,285.0
25%,606.0,40.0,138.0,178.0,270.0,542.0
50%,731.0,40.0,154.0,200.0,294.0,592.0
75%,869.0,333.0,29999.0,214.0,328.0,646.0
max,29999.0,29999.0,29999.0,29999.0,29999.0,29999.0


,p_axis,qrs_axis,t_axis
count,34830.0,34830.0,34830.0
mean,6870.631869,92.460781,141.55234
std,12576.320922,1523.214416,1781.883849
min,-21846.0,-180.0,-180.0
25%,42.0,-17.0,6.0
50%,61.0,15.0,45.0
75%,86.0,50.0,79.0
max,32767.0,29999.0,32767.0


In [30]:
# drop p_onset, p_end, p_axis as predictor because they have so many values at 29999 ms (physiologically impossible) imputing would be dangerous
df = df.drop(columns=['p_onset', 'p_end', 'p_axis'])

# imputing impossible values with median of associated ecg_bucket value
for col in ['rr_interval', 'qrs_onset', 'qrs_end', 't_end']:
    df.loc[df[col]>4000, col] = pd.NA
for col in ['qrs_axis', 't_axis']:
    df.loc[~df[col].between(-180,180), col] = pd.NA

ecg_measurements = ['rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis']
df[ecg_measurements]=df[ecg_measurements].astype(float)
df[ecg_measurements] = df.groupby('ecg_bucket')[ecg_measurements].transform(lambda x: x.fillna(x.median()))

In [ ]:
# handle missing values: language, marital status
# handle spaces and case: language, marital status, race, admission type, admission location, first care unit, last care unit
df['language'] = df['language'].fillna('unknown')
df['language'] = df['language'].str.lower().str.replace(' ', '_')

df['marital_status'] = df['marital_status'].fillna('unknown')
df['marital_status'] = df['marital_status'].str.lower()

df['race'] = df['race'].apply(simplify_race) # function handles

df['admission_type'] = df['admission_type'].str.lower().str.replace(' ','_').str.removesuffix('.')

df['admission_location'] = df['admission_location'].str.lower().str.replace('-', '_').str.replace(' ', '_').str.replace('/', '_')
df['admission_location'].value_counts()

# there are no differences in facilities between first vs last care unit in this dataset, shrinking to 1 column: care_unit
df = df.rename(columns={'first_careunit': 'care_unit'})
df['care_unit'] = df['care_unit'].apply(simplify_careunit)
df = df.drop(columns=['last_careunit'])

# change 'anchor_age' col to 'age'
df = df.rename(columns={'anchor_age': 'age'})

In [34]:
# drop columns that will not be used in modeling
keep_cols = [
    'care_unit', 'admission_type', 'admission_location', 
    'language', 'marital_status', 'race', 'hospital_expire_flag', 
    'gender', 'age', 'rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis', 'ecg_bucket'
]
df = df[keep_cols]
df.shape

(34830, 16)

In [36]:
# Bob: Check columns, head, and export cleaned data to csv
print(df.columns)
display(df.head())
df.to_csv('data/cleaned_data.csv')

Index(['care_unit', 'admission_type', 'admission_location', 'language',
       'marital_status', 'race', 'hospital_expire_flag', 'gender', 'age',
       'rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis',
       'ecg_bucket'],
      dtype='object')


,care_unit,admission_type,admission_location,language,marital_status,race,hospital_expire_flag,gender,age,rr_interval,qrs_onset,qrs_end,t_end,qrs_axis,t_axis,ecg_bucket
0,neuro,ew_emer,physician_referral,english,single,white,0,1,72,434.0,186.0,268.0,484.0,-44.0,119.0,sinus_tachy
1,other,ew_emer,emergency_room,spanish,single,other,0,1,88,869.0,194.0,272.0,658.0,-3.0,64.0,normal_sinus
2,cvicu,urgent,transfer_from_hospital,english,married,other,0,0,85,869.0,202.0,298.0,636.0,0.0,83.0,normal_sinus
3,cvicu,elective,physician_referral,english,married,white,0,0,79,833.0,204.0,344.0,642.0,-37.0,154.0,normal_sinus
4,cvicu,ew_emer,emergency_room,english,single,other,1,0,37,869.0,242.0,348.0,622.0,36.0,38.0,normal_sinus
